# Sketch2Shape API (Kaggle T4)
Runs InstantMesh behind a FastAPI server and publishes it through a free Cloudflare quick tunnel.

**Before running:** Settings -> Accelerator = *GPU T4 x2* (only GPU 0 is used), Internet = *On*.
Run the cells top to bottom. If cell 2 says a restart is needed, restart the kernel and continue from cell 3.
The last cell prints the public URL to paste into the frontend. Keep the session open while using it.

In [ ]:
import shutil, subprocess, requests

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader", "-i", "0"],
                     capture_output=True, text=True).stdout.strip()
assert smi, "No GPU found. Enable a GPU accelerator in the Kaggle settings."
print("GPU 0:", smi)
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Disk free: {free_gb:.1f} GB (need ~12 GB)")
assert free_gb > 12, "Not enough disk space in /kaggle/working"
requests.get("https://github.com", timeout=15).raise_for_status()
print("Internet: OK")

In [ ]:
%pip install -q "pytorch-lightning==2.1.2" omegaconf einops trimesh PyMCubes xatlas imageio[ffmpeg] \
    rembg onnxruntime pymatting fastapi "uvicorn[standard]" python-multipart pillow requests huggingface_hub psutil

import numpy
print("NumPy:", numpy.__version__)

In [ ]:
# Same NumPy pin you used earlier. onnxruntime/rembg are installed as CPU builds on purpose,
# so background removal never takes VRAM from the 3D model.
%pip install -q --no-deps --force-reinstall "numpy==2.1.3"
import numpy
if numpy.__version__ != "2.1.3":
    print("NumPy changed on disk. Restart the kernel (Run -> Restart), then continue from the next cell.")
else:
    print("NumPy OK")

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/kaggle/working/InstantMesh")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TencentARC/InstantMesh.git", str(REPO)], check=True)
print("run.py present:", (REPO / "run.py").exists())

try:
    import nvdiffrast.torch  # noqa: F401
    print("nvdiffrast: already installed")
except Exception:
    r = subprocess.run(["pip", "install", "-q", "--no-cache-dir", "--no-build-isolation", "--no-deps",
                        "git+https://github.com/NVlabs/nvdiffrast.git"])
    print("nvdiffrast install exit code:", r.returncode)

In [ ]:
# Only the base checkpoint is downloaded (the large one is not needed and wastes disk).
# local_dir avoids keeping a second copy in the Hugging Face cache.
from huggingface_hub import hf_hub_download

ckpt_dir = REPO / "ckpts"
ckpt_dir.mkdir(exist_ok=True)
for name in ["diffusion_pytorch_model.bin", "instant_mesh_base.ckpt"]:
    if (ckpt_dir / name).exists():
        print("have", name)
        continue
    hf_hub_download("TencentARC/InstantMesh", name, local_dir=str(ckpt_dir))
    print("downloaded", name)
for p in ckpt_dir.glob("*"):
    if p.is_file():
        print(p.name, f"{p.stat().st_size / 1024**3:.2f} GB")

In [ ]:
# Idempotent compatibility patches (the same fixes you found by hand).
run_py = REPO / "run.py"
src = run_py.read_text()
if "trust_remote_code=True" not in src:
    old = '    torch_dtype=torch.float16,\n)'
    if old in src:
        run_py.write_text(src.replace(old, '    torch_dtype=torch.float16,\n    trust_remote_code=True,\n)', 1))
        print("run.py: patched trust_remote_code")
    else:
        print("run.py: pattern not found, check the DiffusionPipeline.from_pretrained call manually")
else:
    print("run.py: already patched")

dino = REPO / "src/models/encoder/dino.py"
src = dino.read_text()
if "except ImportError:" not in src:
    old = "from transformers.pytorch_utils import find_pruneable_heads_and_indices, prune_linear_layer"
    new = '''from transformers.pytorch_utils import prune_linear_layer

try:
    from transformers.pytorch_utils import find_pruneable_heads_and_indices
except ImportError:
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        heads = set(heads) - already_pruned_heads
        mask = torch.ones(n_heads, head_size, dtype=torch.bool)
        heads = {h - sum(1 for p in already_pruned_heads if p < h) for h in heads}
        for head in heads:
            mask[head] = False
        index = torch.arange(mask.numel())[mask.view(-1)]
        return heads, index'''
    if old in src:
        src = src.replace(old, new, 1)
    else:
        print("dino.py: import line not found")
dino_old = "head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)"
if dino_old in src:
    # Newer transformers removed get_head_mask. Inference always passes head_mask=None.
    src = src.replace(dino_old, "head_mask = [None] * self.config.num_hidden_layers", 1)
    print("dino.py: head_mask patched")
elif "[None] * self.config.num_hidden_layers" in src:
    print("dino.py: head_mask already patched")
else:
    print("dino.py: head_mask line not found, check manually")
dino.write_text(src)
print("dino.py: patched")

## Sketch stage (pencil sketch -> shaded object image)
InstantMesh is trained on rendered objects, so raw line art reconstructs poorly. Sketches first go through
line cleanup + Stable Diffusion 1.5 with a lineart ControlNet (fp16, ~3-4 GB VRAM, separate process), then InstantMesh.
Only the fp16 weights are downloaded (~2.7 GB, into the Hugging Face cache, not /kaggle/working).

In [ ]:
import importlib, subprocess
try:
    import cv2
    print("OpenCV:", cv2.__version__)
except ImportError:
    subprocess.run(["pip", "install", "-q", "--no-deps", "opencv-python-headless"], check=True)
    print("Installed opencv-python-headless")

from huggingface_hub import snapshot_download
snapshot_download("stable-diffusion-v1-5/stable-diffusion-v1-5", allow_patterns=[
    "model_index.json", "scheduler/*", "tokenizer/*", "feature_extractor/*",
    "text_encoder/config.json", "text_encoder/model.fp16.safetensors",
    "unet/config.json", "unet/diffusion_pytorch_model.fp16.safetensors",
    "vae/config.json", "vae/diffusion_pytorch_model.fp16.safetensors",
])
snapshot_download("lllyasviel/control_v11p_sd15_lineart",
                  allow_patterns=["config.json", "diffusion_pytorch_model.fp16.safetensors"])
print("Sketch models ready")

In [ ]:
%%writefile /kaggle/working/sketch2img.py
"""Pencil sketch -> clean line art -> shaded object image (SD 1.5 + lineart ControlNet).

Runs as its own subprocess on GPU 0 and exits before InstantMesh starts, so the two
models never share VRAM. Peak use is roughly 3-4 GB in fp16 with attention slicing.

  python sketch2img.py <sketch.png> --lineart lineart.png --out concept.png [--prompt ...]
  python sketch2img.py <sketch.png> --lineart lineart.png --lines-only   # CPU, no model
"""
import argparse

import cv2
import numpy as np
from PIL import Image

SD_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CN_MODEL = "lllyasviel/control_v11p_sd15_lineart"
SIZE = 512

POSITIVE = (
    "{subject}, a single solid physical object, 3D product render, whole object in view, centered, "
    "plain pure white background, soft even studio lighting, realistic materials and colour, high detail"
)
NEGATIVE = (
    "sketch, drawing, pencil, lineart, outline, monochrome, grayscale, paper, multiple objects, cropped, "
    "cut off, text, watermark, signature, busy background, floor, harsh shadow, blurry, low quality, deformed"
)


def clean_lines(src: str, dest: str) -> None:
    """Turn a photo/scan of a pencil sketch into crisp white-on-black line art.

    1. Flat-field correction removes uneven lighting and paper shading.
    2. Adaptive threshold picks out pencil strokes, even faint ones.
    3. Tiny specks (paper grain, eraser dust) are dropped; small stroke gaps are closed.
    4. The drawing is cropped to its strokes and centred on a square canvas.
    """
    gray = cv2.imread(src, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise SystemExit("Could not read the sketch image")
    h, w = gray.shape
    scale = 1024 / max(h, w)
    if scale < 1:
        gray = cv2.resize(gray, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    h, w = gray.shape

    background = cv2.GaussianBlur(gray, (0, 0), sigmaX=max(h, w) / 25)
    flat = cv2.divide(gray, background, scale=255)
    flat = cv2.GaussianBlur(flat, (3, 3), 0)

    block = max(15, (max(h, w) // 40) | 1)
    lines = cv2.adaptiveThreshold(flat, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, block, 8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(lines, connectivity=8)
    min_area = max(12, int(h * w * 0.00004))
    keep = np.zeros(n, dtype=bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
    lines = np.where(keep[labels], 255, 0).astype(np.uint8)
    lines = cv2.morphologyEx(lines, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))

    ys, xs = np.nonzero(lines)
    if len(xs) < 50:
        raise SystemExit("No pencil lines found. Try a darker or higher-contrast sketch.")
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    crop = lines[y0:y1 + 1, x0:x1 + 1]
    side = int(max(crop.shape) / 0.82)  # ~9% margin on each side, like a product shot
    canvas = np.zeros((side, side), np.uint8)
    oy, ox = (side - crop.shape[0]) // 2, (side - crop.shape[1]) // 2
    canvas[oy:oy + crop.shape[0], ox:ox + crop.shape[1]] = crop

    canvas = cv2.resize(canvas, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    # Normalise stroke width so tiny or huge drawings give ControlNet similar lines.
    canvas = cv2.dilate((canvas > 60).astype(np.uint8) * 255, np.ones((2, 2), np.uint8))
    Image.fromarray(canvas).convert("RGB").save(dest)


def render(lineart: str, out: str, prompt: str, seed: int, steps: int, fidelity: float) -> None:
    import torch
    from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

    def load(cls, repo, **kw):
        try:
            return cls.from_pretrained(repo, torch_dtype=torch.float16, variant="fp16", **kw)
        except (OSError, ValueError):
            return cls.from_pretrained(repo, torch_dtype=torch.float16, **kw)

    print("Loading ControlNet (lineart) and Stable Diffusion 1.5", flush=True)
    controlnet = load(ControlNetModel, CN_MODEL)
    pipe = load(StableDiffusionControlNetPipeline, SD_MODEL, controlnet=controlnet,
                safety_checker=None, requires_safety_checker=False)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_attention_slicing()
    pipe.to("cuda")
    pipe.set_progress_bar_config(disable=True)

    subject = prompt.strip() or "an object"
    print(f"Rendering sketch: seed={seed} steps={steps} fidelity={fidelity}", flush=True)
    image = pipe(
        POSITIVE.format(subject=subject),
        image=Image.open(lineart).convert("RGB"),
        negative_prompt=NEGATIVE,
        num_inference_steps=steps,
        guidance_scale=7.5,
        controlnet_conditioning_scale=fidelity,
        generator=torch.Generator("cuda").manual_seed(seed),
    ).images[0]
    image.save(out)
    print(f"Peak VRAM in sketch stage: {torch.cuda.max_memory_allocated() / 1024**2:.0f} MB", flush=True)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("sketch")
    ap.add_argument("--lineart", required=True)
    ap.add_argument("--out")
    ap.add_argument("--prompt", default="")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--steps", type=int, default=25)
    ap.add_argument("--fidelity", type=float, default=1.0)
    ap.add_argument("--lines-only", action="store_true")
    a = ap.parse_args()
    print("Cleaning up sketch lines", flush=True)
    clean_lines(a.sketch, a.lineart)
    if not a.lines_only:
        render(a.lineart, a.out, a.prompt, a.seed, a.steps, a.fidelity)

## API server
`server.py` is written to disk and started as a separate process, so the notebook stays free.
Each request runs InstantMesh in its own subprocess on GPU 0 and exits afterwards, which releases all VRAM.

In [ ]:
%%writefile /kaggle/working/server.py
"""Sketch2Shape API: wraps InstantMesh for a Kaggle T4 (16 GB VRAM, ~29 GB RAM, 20 GB disk).

Resource safeguards
- One GPU job at a time (single worker thread, bounded queue).
- Every job runs `run.py` in a fresh subprocess pinned to GPU 0, so all VRAM is
  released when the job ends and nothing accumulates between requests.
- Inputs are downscaled, uploads are size-capped, jobs have a hard timeout and
  old job folders are deleted so /kaggle/working stays small.
"""
import io
import os
import queue
import secrets
import shutil
import subprocess
import threading
import time
import uuid
import zipfile
import re
from pathlib import Path

import numpy as np
import trimesh
from fastapi import FastAPI, File, Form, Header, HTTPException, Query, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from PIL import Image, ImageFilter, ImageOps

REPO = Path("/kaggle/working/InstantMesh")
CONFIG = REPO / "configs" / "instant-mesh-base.yaml"  # base model: fits a T4 comfortably
JOBS_DIR = Path("/kaggle/working/jobs")
JOBS_DIR.mkdir(parents=True, exist_ok=True)

TOKEN = os.environ.get("API_TOKEN", "")
MAX_UPLOAD_BYTES = 8 * 1024 * 1024
MAX_QUEUE = 3
KEEP_JOBS = 4
JOB_TIMEOUT_S = 900
INPUT_SIZE = 512
SKETCH_SCRIPT = Path("/kaggle/working/sketch2img.py")
INPUT_TYPES = {"auto", "sketch", "photo"}

app = FastAPI(title="Sketch2Shape API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

jobs: dict[str, dict] = {}
work_queue: "queue.Queue[str]" = queue.Queue()


def check_auth(header_token: str | None, query_token: str | None) -> None:
    if not TOKEN:
        return
    supplied = header_token or query_token or ""
    if not secrets.compare_digest(supplied, TOKEN):
        raise HTTPException(status_code=401, detail="Invalid or missing API token")


def gpu_status() -> dict:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.used,memory.total",
             "--format=csv,noheader,nounits", "-i", "0"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip()
        name, used, total = [p.strip() for p in out.split(",")]
        return {"name": name, "memory_used_mb": int(used), "memory_total_mb": int(total)}
    except Exception:
        return {"name": None, "memory_used_mb": None, "memory_total_mb": None}


def prepare_image(raw: bytes, dest: Path, enhance: bool, size: int = INPUT_SIZE) -> None:
    """Flatten transparency onto white, square-pad, downscale, optionally boost contrast."""
    img = Image.open(io.BytesIO(raw))
    img = ImageOps.exif_transpose(img)
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(bg, img)
    img = img.convert("RGB")
    img.thumbnail((size, size), Image.LANCZOS)
    side = max(img.size)
    canvas = Image.new("RGB", (side, side), (255, 255, 255))
    canvas.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    if enhance:
        canvas = ImageOps.autocontrast(canvas, cutoff=1)
    canvas.save(dest, format="PNG")


def looks_like_sketch(path: Path) -> bool:
    """Pencil sketches are nearly colourless and mostly blank paper.

    Lighting is flattened first (divide by a heavy blur), so a dim or unevenly lit
    phone photo of paper still reads as mostly white. Then: low colour saturation and
    only a small share of "ink" pixels.
    """
    img = Image.open(path).convert("RGB").resize((256, 256))
    a = np.asarray(img, dtype=np.float32) / 255
    saturation = (a.max(axis=2) - a.min(axis=2)).mean()
    gray = np.asarray(img.convert("L"), dtype=np.float32)
    background = np.asarray(img.convert("L").filter(ImageFilter.GaussianBlur(12)), dtype=np.float32)
    flat = np.clip(gray / np.maximum(background, 1), 0, 1.5)
    ink = (flat < 0.8).mean()
    paper = (flat > 0.92).mean()  # textured grey photos fail this; blank paper passes
    return bool(saturation < 0.12 and ink < 0.2 and paper > 0.6)


class Profiler:
    """Samples GPU 0 VRAM and RAM every 0.5 s while a job runs, and timestamps log lines.

    Log lines (from run.py) and server stage changes become timeline events, so a
    peak can be matched to the step that caused it. tqdm progress bars are skipped.
    """

    INTERVAL = 0.5
    MAX_SAMPLES = 400
    MAX_EVENTS = 60

    def __init__(self, log_path: Path):
        self.log_path = log_path
        self.t0 = time.time()
        self.samples: list[list[float]] = []  # [t, vram_mb, job_ram_mb, sys_ram_mb]
        self.events: list[list] = []  # [t, text]
        self.lock = threading.Lock()
        self.stop_flag = threading.Event()
        self.log_pos = 0
        self.root_pid: int | None = None
        self.baseline_vram = self._vram()
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    @staticmethod
    def _vram() -> float:
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                capture_output=True, text=True, timeout=5,
            ).stdout.strip()
            return float(out)
        except Exception:
            return 0.0

    @staticmethod
    def _sys_ram() -> tuple[float, float]:
        info = {}
        with open("/proc/meminfo") as f:
            for line in f:
                k, v = line.split(":", 1)
                info[k] = int(v.split()[0]) / 1024
        return info["MemTotal"] - info.get("MemAvailable", info["MemFree"]), info["MemTotal"]

    def _job_ram(self) -> float:
        """RSS of the running inference process and its children."""
        if not self.root_pid:
            return 0.0
        try:
            import psutil
            root = psutil.Process(self.root_pid)
            procs = [root] + root.children(recursive=True)
            return sum(p.memory_info().rss for p in procs) / 1024**2
        except Exception:
            return 0.0

    def mark(self, text: str) -> None:
        with self.lock:
            if len(self.events) < self.MAX_EVENTS:
                self.events.append([round(time.time() - self.t0, 1), text[:100]])

    def _read_log(self) -> None:
        try:
            with open(self.log_path, "rb") as f:
                f.seek(self.log_pos)
                chunk = f.read()
                self.log_pos = f.tell()
        except FileNotFoundError:
            return
        for line in re.split(r"[\r\n]+", chunk.decode(errors="ignore")):
            line = line.strip()
            if not line or "%|" in line or "it/s" in line or "s/it" in line or line.startswith("$ "):
                continue
            if line.startswith(("Traceback", "File ", "  ")) and "Error" not in line:
                continue
            self.mark(line)

    def _loop(self) -> None:
        while not self.stop_flag.is_set():
            vram = self._vram()
            sys_used, _ = self._sys_ram()
            with self.lock:
                self.samples.append([round(time.time() - self.t0, 1), vram, round(self._job_ram()), round(sys_used)])
                if len(self.samples) > self.MAX_SAMPLES:
                    self.samples = self.samples[::2]  # halve resolution, keep whole timeline
            self._read_log()
            self.stop_flag.wait(self.INTERVAL)

    def stop(self) -> None:
        self.stop_flag.set()
        self.thread.join(timeout=5)
        self._read_log()

    def snapshot(self) -> dict:
        _, ram_total = self._sys_ram()
        with self.lock:
            samples = [list(s) for s in self.samples]
            events = [list(e) for e in self.events]
        def peak(i):
            if not samples:
                return {"mb": 0, "t": 0}
            best = max(samples, key=lambda s: s[i])
            return {"mb": round(best[i]), "t": best[0]}
        return {
            "samples": samples,
            "events": events,
            "gpu_total_mb": gpu_status()["memory_total_mb"],
            "ram_total_mb": round(ram_total),
            "baseline_vram_mb": round(self.baseline_vram),
            "peak_vram": peak(1),
            "peak_job_ram": peak(2),
            "peak_sys_ram": peak(3),
        }


def set_state(job: dict, status: str, stage: str, **extra) -> None:
    job.update(status=status, stage=stage, updated=time.time(), **extra)
    if job.get("_prof"):
        job["_prof"].mark(f"[server] {stage}")


def run_cmd(job: dict, cmd: list[str], cwd: Path, log_path: Path) -> int:
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    with open(log_path, "ab") as log:
        log.write(f"\n$ {' '.join(cmd)}\n".encode())
        log.flush()
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
        if job.get("_prof"):
            job["_prof"].root_pid = proc.pid
        try:
            return proc.wait(timeout=JOB_TIMEOUT_S)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
            return -9


def instantmesh_cmd(job: dict, image_path: Path, out_dir: Path, texture: bool, remove_bg: bool) -> list[str]:
    cmd = [
        "python", "run.py", str(CONFIG), str(image_path),
        "--output_path", str(out_dir),
        "--view", "6",  # use all six Zero123++ views
        "--diffusion_steps", str(job["params"]["steps"]),
    ]
    if not remove_bg:
        cmd.append("--no_rembg")
    if texture:
        cmd.append("--export_texmap")
    return cmd


def last_log_line(log_path: Path) -> str:
    lines = [l.strip() for l in log_path.read_text(errors="ignore").splitlines() if l.strip()]
    return lines[-1] if lines else "unknown error"


def export_assets(out_dir: Path, job_dir: Path) -> dict:
    objs = sorted(out_dir.rglob("meshes/*.obj"))
    if not objs:
        raise RuntimeError("InstantMesh finished but no OBJ mesh was produced")
    obj = objs[0]
    mesh = trimesh.load(obj, force="mesh")
    glb = job_dir / "model.glb"
    mesh.export(glb, file_type="glb")

    # Validate: the GLB must reopen and contain real geometry.
    check = trimesh.load(glb, force="mesh")
    if len(check.faces) == 0:
        raise RuntimeError("Exported GLB has no faces")

    bundle = job_dir / "model_obj.zip"
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
        for f in obj.parent.iterdir():
            if f.stem == obj.stem or f.suffix.lower() in {".mtl", ".png", ".jpg"}:
                z.write(f, arcname=f.name)
    return {
        "vertices": int(len(check.vertices)),
        "faces": int(len(check.faces)),
        "watertight": bool(check.is_watertight),
        "textured": bool(getattr(check.visual, "kind", None) == "texture"),
        "glb_mb": round(glb.stat().st_size / 1024**2, 2),
    }


def process(job_id: str) -> None:
    job = jobs[job_id]
    job_dir = JOBS_DIR / job_id
    out_dir = job_dir / "out"
    log_path = job_dir / "log.txt"
    image_path = job_dir / "input.png"
    started = time.time()
    job["_prof"] = Profiler(log_path)
    try:
        params = job["params"]
        kind = params["input_type"]
        if kind == "auto":
            kind = "sketch" if looks_like_sketch(job_dir / "sketch_raw.png") else "photo"
        job["input_type"] = kind
        recon_input, remove_bg = image_path, params["remove_bg"]

        if kind == "sketch":
            set_state(job, "running", "Cleaning sketch lines and rendering the object")
            code = run_cmd(job, [
                "python", str(SKETCH_SCRIPT), str(job_dir / "sketch_raw.png"),
                "--lineart", str(job_dir / "lineart.png"), "--out", str(job_dir / "concept.png"),
                "--prompt", params["prompt"], "--seed", str(params["seed"]),
                "--fidelity", str(params["fidelity"]),
            ], JOBS_DIR.parent, log_path)
            if (job_dir / "lineart.png").exists():
                job["artifacts"].append("lineart")
            if code != 0:
                if code == -9:
                    raise RuntimeError(f"Sketch stage exceeded the {JOB_TIMEOUT_S}s time limit")
                raise RuntimeError("Sketch stage failed: " + last_log_line(log_path))
            job["artifacts"].append("concept")
            # The render sits on white; rembg trims and re-centres it the way Zero123++ expects.
            recon_input, remove_bg = job_dir / "concept.png", True

        set_state(job, "running", "Generating multi-view images and reconstructing mesh")
        want_texture = params["texture"]
        code = run_cmd(job, instantmesh_cmd(job, recon_input, out_dir, want_texture, remove_bg), REPO, log_path)
        if code != 0 and want_texture and code != -9:
            set_state(job, "running", "Texture export failed, retrying without texture map")
            shutil.rmtree(out_dir, ignore_errors=True)
            code = run_cmd(job, instantmesh_cmd(job, recon_input, out_dir, False, remove_bg), REPO, log_path)
        if code == -9:
            raise RuntimeError(f"Job exceeded the {JOB_TIMEOUT_S}s time limit")
        if code != 0:
            tail = log_path.read_text(errors="ignore")[-3000:]
            if "out of memory" in tail.lower():
                raise RuntimeError("GPU ran out of memory. Try fewer diffusion steps or a simpler image.")
            raise RuntimeError("InstantMesh failed. Log tail:\n" + tail[-800:])
        set_state(job, "running", "Exporting and validating GLB")
        stats = export_assets(out_dir, job_dir)
        shutil.rmtree(out_dir, ignore_errors=True)  # keep disk small; GLB + zip are kept
        set_state(job, "done", "Finished", stats=stats, seconds=round(time.time() - started, 1))
    except Exception as exc:  # noqa: BLE001
        set_state(job, "error", "Failed", error=str(exc))
    finally:
        prof = job.pop("_prof")
        prof.stop()
        job["profile"] = prof.snapshot()  # frozen copy kept with the result


def worker() -> None:
    while True:
        job_id = work_queue.get()
        try:
            process(job_id)
        finally:
            work_queue.task_done()
            prune()


def prune() -> None:
    finished = sorted(
        (j for j in jobs.values() if j["status"] in ("done", "error")),
        key=lambda j: j["created"],
    )
    for old in finished[:-KEEP_JOBS]:
        shutil.rmtree(JOBS_DIR / old["id"], ignore_errors=True)
        jobs.pop(old["id"], None)


threading.Thread(target=worker, daemon=True).start()


@app.get("/health")
def health(x_api_token: str | None = Header(default=None), token: str | None = Query(default=None)):
    check_auth(x_api_token, token)
    return {
        "ok": True,
        "model": "instant-mesh-base",
        "gpu": gpu_status(),
        "queued": work_queue.qsize(),
        "disk_free_gb": round(shutil.disk_usage("/kaggle/working").free / 1024**3, 1),
    }


@app.post("/jobs")
async def create_job(
    file: UploadFile = File(...),
    steps: int = Form(40),
    remove_bg: bool = Form(False),
    texture: bool = Form(True),
    enhance: bool = Form(True),
    input_type: str = Form("auto"),
    prompt: str = Form(""),
    fidelity: float = Form(1.0),
    seed: int | None = Form(None),
    x_api_token: str | None = Header(default=None),
    token: str | None = Query(default=None),
):
    check_auth(x_api_token, token)
    if input_type not in INPUT_TYPES:
        raise HTTPException(status_code=400, detail=f"Unknown input type {input_type!r}")
    if work_queue.qsize() >= MAX_QUEUE:
        raise HTTPException(status_code=429, detail="Queue is full, try again shortly")
    raw = await file.read(MAX_UPLOAD_BYTES + 1)
    if len(raw) > MAX_UPLOAD_BYTES:
        raise HTTPException(status_code=413, detail="Image larger than 8 MB")
    job_id = uuid.uuid4().hex[:12]
    job_dir = JOBS_DIR / job_id
    job_dir.mkdir(parents=True)
    try:
        prepare_image(raw, job_dir / "input.png", enhance)
        prepare_image(raw, job_dir / "sketch_raw.png", enhance=False, size=1536)  # full detail for line cleanup
    except Exception:
        shutil.rmtree(job_dir, ignore_errors=True)
        raise HTTPException(status_code=400, detail="Could not read that image file")
    job_seed = seed if seed is not None else secrets.randbelow(2**31)
    jobs[job_id] = {
        "id": job_id, "status": "queued", "stage": "Waiting for GPU",
        "created": time.time(), "updated": time.time(),
        "input_type": input_type, "artifacts": [],
        "seed": job_seed,
        "params": {"steps": max(20, min(int(steps), 50)), "remove_bg": remove_bg, "texture": texture,
                   "input_type": input_type, "prompt": prompt.strip()[:300],
                   "seed": job_seed, "fidelity": round(max(0.4, min(float(fidelity), 1.6)), 2)},
    }
    work_queue.put(job_id)
    return {"id": job_id}


@app.get("/jobs/{job_id}")
def get_job(job_id: str, x_api_token: str | None = Header(default=None),
            token: str | None = Query(default=None)):
    check_auth(x_api_token, token)
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Unknown job")
    out = {k: v for k, v in job.items() if k not in ("params", "_prof")}
    if job.get("_prof"):
        out["profile"] = job["_prof"].snapshot()  # live while running
    return out | {"elapsed": round(time.time() - job["created"], 1)}


@app.get("/jobs/{job_id}/{kind}")
def download(job_id: str, kind: str, x_api_token: str | None = Header(default=None),
             token: str | None = Query(default=None)):
    check_auth(x_api_token, token)
    job = jobs.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Unknown job")
    files = {
        "glb": ("model.glb", "model/gltf-binary", True),
        "obj": ("model_obj.zip", "application/zip", True),
        "lineart": ("lineart.png", "image/png", False),  # intermediates can be fetched while running
        "concept": ("concept.png", "image/png", False),
    }
    if kind not in files:
        raise HTTPException(status_code=404, detail="Unknown file type")
    name, mime, needs_done = files[kind]
    path = JOBS_DIR / job_id / name
    if (needs_done and job["status"] != "done") or not path.exists():
        raise HTTPException(status_code=404, detail="Result not ready")
    return FileResponse(path, media_type=mime, filename=name)

In [ ]:
import os, secrets, subprocess, time, requests

API_TOKEN = secrets.token_urlsafe(12)
SERVER_LOG = "/kaggle/working/server.log"
server = None

def server_healthy():
    try:
        return requests.get("http://127.0.0.1:8000/health", headers={"X-API-Token": API_TOKEN}, timeout=3).ok
    except Exception:
        return False

def start_server():
    """(Re)start uvicorn detached from the kernel so interrupts do not kill it."""
    global server
    if server is not None and server.poll() is None:
        server.terminate()
    env = {**os.environ, "API_TOKEN": API_TOKEN, "CUDA_VISIBLE_DEVICES": "0"}
    server = subprocess.Popen(
        ["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
        cwd="/kaggle/working", env=env, start_new_session=True,
        stdout=open(SERVER_LOG, "ab"), stderr=subprocess.STDOUT,
    )
    for _ in range(40):
        if server_healthy():
            return True
        if server.poll() is not None:
            break
        time.sleep(1)
    print("Server failed to start. Log tail:\n", open(SERVER_LOG).read()[-3000:])
    return False

assert start_server(), "API did not start"
print("API is up")

In [ ]:
import re, threading, urllib.request
from pathlib import Path

cf = Path("/kaggle/working/cloudflared")
if not cf.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    cf.chmod(0o755)

tunnel = subprocess.Popen(
    [str(cf), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
def _watch():
    global public_url
    for line in tunnel.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m and not public_url:
            public_url = m.group(0)
threading.Thread(target=_watch, daemon=True).start()
for _ in range(60):
    if public_url:
        break
    time.sleep(1)
assert public_url, "Tunnel did not return a URL"

print("=" * 60)
print("PUBLIC API URL :", public_url)
print("API TOKEN      :", API_TOKEN)
print("=" * 60)
print("Paste both into the frontend. Tokens change every time this notebook restarts.")

In [ ]:
# Keep this cell running while you use the frontend. It restarts the API if it dies.
# The tunnel URL stays valid across API restarts.
import time, requests
try:
    while True:
        if tunnel.poll() is not None:
            print("TUNNEL STOPPED. Re-run the tunnel cell to get a new public URL.")
        try:
            h = requests.get("http://127.0.0.1:8000/health", headers={"X-API-Token": API_TOKEN}, timeout=5).json()
            g = h["gpu"]
            print(f"{time.strftime('%H:%M:%S')}  queue={h['queued']}  VRAM {g['memory_used_mb']}/{g['memory_total_mb']} MB  disk free {h['disk_free_gb']} GB")
        except Exception as exc:
            print(f"{time.strftime('%H:%M:%S')}  API is down ({type(exc).__name__}); exit code: {server.poll()}")
            print("--- server.log tail ---")
            print(open(SERVER_LOG).read()[-1500:])
            print("--- restarting ---")
            start_server()
        time.sleep(30)
except KeyboardInterrupt:
    print("Monitor stopped. API and tunnel are still running.")